# k5twin on Google Colab

k5twin is a software twin of the NXP S32K566 Cortex-M7 + eIQ Auto target. It runs a compiled eIQ Auto (Glow) network on an emulated Cortex-M7 and reports outputs, memory use and predicted board time.

It is part of [Internal thermal scanner of 4680 cells with NXP S32K5 BMS](https://github.com/RemusTeodorescu/4680-cell-models/tree/internal-thermal-scanner-s32k5/Internal-thermal-scanner-of-4680-cells-with-NXP-S32K5-BMS).

The twin itself is fetched from GitHub. The compiled networks it runs are made with NXP's licensed eIQ Auto toolchain and are not in the public repository, so you supply your own eIQ Auto project folders as zips, either from Google Drive (default `MyDrive/k5twin_colab/`) or by upload:
- a project with the compiled networks, e.g. `05_S32K5_Temperature_estimation_200_cells.zip` (required for cells 4–8);
- optionally `04_S32K5_Temperature_estimation_scaling.zip` (recalibration, cell 9).

Run the cells from top to bottom (Runtime → Run all). A CPU runtime is enough.

In [ ]:
# 1. The twin from GitHub, and your eIQ Auto project zips from Drive (or set USE_UPLOAD = True)
!git clone -q --depth 1 -b internal-thermal-scanner-s32k5 https://github.com/RemusTeodorescu/4680-cell-models /content/repo
K5 = '/content/repo/Internal-thermal-scanner-of-4680-cells-with-NXP-S32K5-BMS/k5twin'
USE_UPLOAD = False
DRIVE_DIR = '/content/drive/MyDrive/k5twin_colab'

import os, glob, shutil, zipfile
os.makedirs('/content/in', exist_ok=True)
if USE_UPLOAD:
    from google.colab import files
    for name in files.upload():
        shutil.move(name, '/content/in/' + name)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    for z in glob.glob(DRIVE_DIR + '/*.zip'):
        shutil.copy(z, '/content/in/')
print(os.listdir('/content/in'))

In [ ]:
# 2. Tools: QEMU (Cortex-M7 emulator), ARM GCC + newlib, GDB, Python packages (about 1-2 min)
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y qemu-system-arm gcc-arm-none-eabi libnewlib-arm-none-eabi gdb-multiarch > /dev/null
!pip -q install ai-edge-litert scipy
!qemu-system-arm --version | head -1; arm-none-eabi-gcc --version | head -1
# The twin needs the mps2-an500 (Cortex-M7) board, QEMU >= 6.0.
import subprocess, os
if 'mps2-an500' not in subprocess.run(['qemu-system-arm','-machine','help'],capture_output=True,text=True).stdout:
    print('apt QEMU has no Cortex-M7 board -> using xPack QEMU 8.2.2')
    !wget -q https://github.com/xpack-dev-tools/qemu-arm-xpack/releases/download/v8.2.2-1/xpack-qemu-arm-8.2.2-1-linux-x64.tar.gz -O /tmp/xq.tgz
    !mkdir -p /opt/xq && tar -xzf /tmp/xq.tgz -C /opt/xq
    os.environ['K5_QEMU'] = glob.glob('/opt/xq/*/bin/qemu-system-arm')[0]
print('QEMU:', os.environ.get('K5_QEMU', 'qemu-system-arm'))

In [ ]:
# 3. Unpack the eIQ Auto projects
for z in glob.glob('/content/in/*.zip'):
    zipfile.ZipFile(z).extractall('/content/' + os.path.splitext(os.path.basename(z))[0])
P200 = (glob.glob('/content/*200_cells*') or ['/content'])[0]
print('project:', P200)
%cd $K5

In [ ]:
# 4. Self-test: the emulated M7 must reproduce the board results (CPU_vs_NPU report, Table 2)
!python3 tests/selftest.py "$P200"

In [ ]:
# 5. Arnold's 200-cell CPU build: outputs, instructions, predicted time, memory
import glob
cpu_h = glob.glob(P200 + '/**/data/cpu_glow/models/*.h', recursive=True)[0]
app = cpu_h.split('/data/')[0]
elf = glob.glob(app + '/build_cpu_glow*/bin/*cpu_glow')[0]
!python3 k5twin.py run "$cpu_h" --inputs examples/hev2_5x40_batch200.json --board-elf "$elf" --board-map "$elf.map"
print('\nThe same with the faster expf linked in:')
!python3 k5twin.py run "$cpu_h" --inputs examples/hev2_5x40_batch200.json --link opt/fast_expf.c | head -10
# Expected with GCC 13.2 / newlib 4.4: 976,969 instructions, 9.69 ms.
# A different toolchain version changes the expf instruction count a little.

In [ ]:
# 6. Arnold's 200-cell NPU (Neutron) build
proj = app.rsplit('/eiqa_runtime_app', 1)[0]
npu_h = glob.glob(app + '/data/neutron/models/*.h')[0]
tfl = glob.glob(proj + '/glow_model_data_test/subgraph_0/transform/*_p0.tflite')[0]
!python3 k5twin.py npu "$npu_h" --inputs examples/hev2_5x40_batch200.json --layers 3 --neutron-log "$proj/eiqa_offline/eiqa.log" --tflite "$tfl"

In [ ]:
# 7. The GUI: starts the local server and opens it in a new browser tab through Colab's proxy
import subprocess, time
gui = subprocess.Popen(['python3', 'k5twin_gui.py', '--root', P200, '--port', '8765', '--no-browser'])
time.sleep(3)
from google.colab import output
output.serve_kernel_port_as_window(8765)
# If the link does not open, use: output.serve_kernel_port_as_iframe(8765, height=1100)

### Debugging in Colab
Colab has no interactive terminal on the free tier, so GDB runs as a batch script here. On your PC (WSL), the GUI's **Start debugger** button and an interactive `gdb-multiarch` are more convenient.

In [ ]:
# 8. Example GDB session: stop at the bundle entry, show inputs; stop at the end, show the first outputs
import subprocess, time, os
q = subprocess.Popen(['python3', 'k5twin.py', 'debug', cpu_h, '--inputs', 'examples/hev2_5x40_batch200.json', '--work', '/tmp/k5dbg', '--port', '3333'])
time.sleep(3)
!gdb-multiarch -q -batch /tmp/k5dbg/twin.elf -ex 'target remote :3333' -ex 'break trained_PINN_model_p0_m7' -ex 'break k5_mark_end' -ex continue -ex 'x/5fw mutableWeight+832' -ex continue -ex 'x/5fw mutableWeight+4864' -ex kill 2>&1 | grep -v '^warning'
q.terminate()

In [ ]:
# 9. Optional: recalibrate on Arnold's 28 measured runs (needs 04_S32K5_Temperature_estimation_scaling.zip)
import json
sc = glob.glob('/content/04_S32K5_Temperature_estimation_scaling/**/eiqa_runtime_app/data', recursive=True)
if sc:
    m = json.load(open('calibration/measured_sweeps.json'))
    for it in m:
        it['bundle'] = sc[0] + '/' + it['bundle'].split('/eiqa_runtime_app/data/')[1]
        it['inputs'] = os.path.abspath('examples/hev2_sample1.json')
    json.dump(m, open('/tmp/measured_colab.json', 'w'))
    !python3 k5twin.py calibrate /tmp/measured_colab.json --cal calibration/m7_200MHz_colab.json
else:
    print('scaling zip not uploaded - using the shipped calibration')